# Predicción de parámetros RLWM a partir de métricas conductuales con una ANN

Este notebook genera las figuras y tablas del proyecto. Los cálculos pesados (simulación de 713 millones de agentes, entrenamiento de la red y simulaciones de soluciones múltiples) se hicieron en el clúster Ixachi del CICESE. Aquí solo se leen sus resultados.

Archivos que se leen de la carpeta del proyecto en Google Drive

| Archivo | Contenido |
|---|---|
| `registro_entrenamiento.csv` | Pérdida de entrenamiento y validación durante la selección y el entrenamiento final |
| `seleccion.json` | Resultados del group lasso para cada λ |
| `metricas_prueba.json` | Métricas de la red en los 142,640,950 agentes de prueba |
| `muestra_predicciones_prueba.npz` | Muestra al azar de 200,000 agentes de prueba con parámetros verdaderos y predichos |
| `soluciones_simuladas.csv` | Parte B, 10,000 agentes de prueba con verdad conocida |
| `soluciones_reales_anon.csv` | Parte A, 37 participantes de fase 3 con identificador anónimo (P01 a P37) |
| `muestra_metricas.npz` | 59 métricas de 200,000 agentes al azar, para el análisis de redundancia |

Las figuras se guardan en la subcarpeta `figuras` en PDF (para LaTeX) y PNG.

## 0. Preparación

Monta Google Drive, revisa que estén los archivos y define el estilo de las figuras. Al correr esta celda, Colab pide permiso para acceder a Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

CARPETA = Path('/content/drive/MyDrive/INTRODUCCIÓN A LA CIENCIA DE DATOS/PROYECTO/proyecto_drive')
ARCHIVOS = ['registro_entrenamiento.csv', 'seleccion.json', 'metricas_prueba.json',
            'muestra_predicciones_prueba.npz', 'soluciones_simuladas.csv', 'soluciones_reales_anon.csv',
            'muestra_metricas.npz']
for a in ARCHIVOS:
    print(('OK     ' if (CARPETA / a).exists() else 'FALTA  ') + a)

PARAMS = ['alpha', 'phi', 'rho', 'epsilon', 'K']
SIMBOLO = {'alpha': 'α', 'phi': 'ϕ', 'rho': 'ρ', 'epsilon': 'ε', 'K': 'K'}
UMBRALES = [0.80, 0.90, 0.95]
CLIP = {'alpha': (0.01, 0.99), 'phi': (0.01, 0.99), 'rho': (0.01, 0.99), 'epsilon': (0.01, 0.49), 'K': (2, 4)}
RANGO = {'alpha': 0.98, 'phi': 0.98, 'rho': 0.98, 'epsilon': 0.48, 'K': 2}

FIG = CARPETA / 'figuras'
FIG.mkdir(exist_ok=True)
plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})

def guardar(fig, nombre):
    fig.savefig(FIG / (nombre + '.pdf'), bbox_inches='tight')
    fig.savefig(FIG / (nombre + '.png'), bbox_inches='tight')
    print('Figura guardada en', FIG / nombre, '(.pdf y .png)')

In [ ]:
registro = pd.read_csv(CARPETA / 'registro_entrenamiento.csv')
seleccion = json.load(open(CARPETA / 'seleccion.json', encoding='utf-8'))
metricas = json.load(open(CARPETA / 'metricas_prueba.json', encoding='utf-8'))

muestra = np.load(CARPETA / 'muestra_predicciones_prueba.npz')
verdadero, predicho = muestra['verdadero'], muestra['predicho']
assert [str(o) for o in muestra['objetivos']] == PARAMS

sim = pd.read_csv(CARPETA / 'soluciones_simuladas.csv')
real = pd.read_csv(CARPETA / 'soluciones_reales_anon.csv')

ruta_m = CARPETA / 'muestra_metricas.npz'
if ruta_m.exists():
    mm = np.load(ruta_m)
    Xm = mm['X'].astype(np.float64)
    nombres_m = [str(n) for n in mm['names']]
else:
    Xm, nombres_m = None, None

print('Registro de entrenamiento', registro.shape)
print('Muestra de prueba        ', verdadero.shape)
print('Parte B, simulados       ', sim.shape)
print('Parte A, reales          ', real.shape)
print('Muestra de métricas      ', None if Xm is None else Xm.shape)

## Figura 1. Curva de entrenamiento

Pérdida (error cuadrático medio de los cinco parámetros escalados de 0 a 1) en entrenamiento y validación, evaluada cada cuarto de pasada. Una pasada es una vuelta completa por los 427,937,930 agentes de entrenamiento, en lotes de 8,192 (52,239 pasos).

El punto rojo marca la red que se guardó. El entrenamiento solo contaba como mejora una baja mayor a 0.01 % sobre el mejor valor anterior, y se detenía tras 3 pasadas sin una mejora así. Por eso la red guardada no es necesariamente el punto más bajo de la curva. El eje vertical está muy acercado, así que la celda imprime cuánto bajó la pérdida en porcentaje y la pérdida de referencia que tendría una red que predijera siempre el promedio.

In [ ]:
fin = registro[registro['etapa'] == 'final'].reset_index(drop=True)

mejor, guardada = np.inf, 0
for i, v in enumerate(fin['perdida_validacion']):
    if v < mejor * (1 - 1e-4):
        mejor, guardada = v, i

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(fin['pasada'], fin['perdida_entrenamiento'], color='0.6', label='Entrenamiento')
ax.plot(fin['pasada'], fin['perdida_validacion'], color='C0', label='Validación')
ax.scatter([fin.loc[guardada, 'pasada']], [fin.loc[guardada, 'perdida_validacion']], color='C3', zorder=3,
           label='Red guardada (pasada %.2f)' % fin.loc[guardada, 'pasada'])
ax.set_xlabel('Pasadas sobre los datos de entrenamiento')
ax.set_ylabel('Pérdida (MSE)')
ax.legend(frameon=False)
fig.tight_layout()
guardar(fig, 'fig1_entrenamiento')
plt.show()

p1 = fin.loc[(fin['pasada'] - 1).abs().idxmin(), 'perdida_validacion']
pg = fin.loc[guardada, 'perdida_validacion']
off = np.array([0, 0, 0, 0, 2.0])
esc = np.array([1, 1, 1, 0.5, 2.0])
referencia = np.mean(np.var((verdadero - off) / esc, axis=0))
print('Validación en la pasada 1         %.5f' % p1)
print('Validación de la red guardada     %.5f (pasada %.2f)' % (pg, fin.loc[guardada, 'pasada']))
print('Baja desde la pasada 1            %.2f %%' % (100 * (1 - pg / p1)))
print('Referencia (predecir el promedio) %.5f' % referencia)
print('Proporción de varianza explicada  %.3f' % (1 - pg / referencia))

## Figura 2. Selección de características con group lasso

Para cada intensidad λ se muestra la pérdida de validación (línea) y el número de métricas que siguen activas (barras). La línea punteada marca la tolerancia (1 % sobre la mejor pérdida). Se elige la λ con menos métricas que queda dentro de la tolerancia.

In [ ]:
res = pd.DataFrame([{'lambda': r['lambda'], 'perdida_validacion': r['perdida_validacion'],
                     'n_seleccionadas': r['n_seleccionadas']} for r in seleccion['resultados']])
mejor_l = res['perdida_validacion'].min()
limite = mejor_l * (1 + seleccion['tolerancia'])
x = np.arange(len(res))

fig, ax = plt.subplots(figsize=(5, 3.2))
ax2 = ax.twinx()
ax2.bar(x, res['n_seleccionadas'], width=0.5, color='C1', alpha=0.25)
ax2.set_ylim(0, res['n_seleccionadas'].max() * 1.3)
ax2.set_ylabel('Métricas activas', color='C1')
ax2.spines['right'].set_visible(True)
ax.set_zorder(ax2.get_zorder() + 1)
ax.patch.set_visible(False)

ax.plot(x, res['perdida_validacion'], marker='o', color='C0')
ax.axhline(limite, color='0.5', lw=1, ls='--', label='Tolerancia (%.0f %% sobre la mejor)' % (100 * seleccion['tolerancia']))
el = list(res['lambda']).index(seleccion['lambda_elegida'])
ax.scatter([el], [res['perdida_validacion'][el]], s=140, facecolors='none', edgecolors='C3', label='λ elegida')
ax.set_xticks(x)
ax.set_xticklabels([str(l) for l in res['lambda']])
ax.set_xlabel('Intensidad del group lasso (λ)')
ax.set_ylabel('Pérdida de validación (MSE)', color='C0')
ax.legend(frameon=False, loc='upper left', fontsize=8)
fig.tight_layout()
guardar(fig, 'fig2_seleccion_lambda')
plt.show()

res['aumento_%'] = 100 * (res['perdida_validacion'] / mejor_l - 1)
print(res.round(5).to_string(index=False))
print('Métricas eliminadas con la λ más alta', seleccion['resultados'][-1]['lambda'], ':',
      [k for k, v in seleccion['resultados'][-1]['norma_por_caracteristica'].items() if v == 0])

## Figura 3 y Tabla 1. Parámetros predichos contra verdaderos

Muestra al azar de 200,000 agentes del conjunto de prueba. En α, ϕ, ρ y ε cada hexágono cuenta agentes (escala logarítmica) y la línea punteada es la identidad, donde caería una predicción perfecta. Si los puntos se aplanan hacia el centro, la red está prediciendo el promedio de los valores compatibles con la conducta. K se muestra como matriz de confusión con proporciones por fila.

La Tabla 1 da intervalos de confianza al 95 % por bootstrap (500 remuestreos de la muestra de 200,000). Con tantos agentes los intervalos son muy estrechos. La variabilidad que importa interpretar es la del error entre agentes, que se ve en la dispersión de la figura.

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(16, 3.3))
for j, p in enumerate(PARAMS[:4]):
    ax = axs[j]
    v, q = verdadero[:, j], predicho[:, j]
    ax.hexbin(v, q, gridsize=40, mincnt=1, bins='log', cmap='viridis')
    lo, hi = min(v.min(), q.min()), max(v.max(), q.max())
    ax.plot([lo, hi], [lo, hi], color='C3', lw=1, ls='--')
    ax.set_title('%s   r = %.3f' % (SIMBOLO[p], metricas[p]['pearson_r']))
    ax.set_xlabel('Verdadero')
    if j == 0:
        ax.set_ylabel('Predicho por la red')

conf = np.array(metricas['K']['confusion_true_rows_pred_cols'], dtype=float)
conf = conf / conf.sum(axis=1, keepdims=True)
ax = axs[4]
ax.imshow(conf, cmap='Blues', vmin=0, vmax=1)
for a in range(3):
    for b in range(3):
        ax.text(b, a, '%.2f' % conf[a, b], ha='center', va='center', color='w' if conf[a, b] > 0.6 else 'k')
ax.set_xticks(range(3))
ax.set_xticklabels(['2', '3', '4'])
ax.set_yticks(range(3))
ax.set_yticklabels(['2', '3', '4'])
ax.set_xlabel('K predicho (redondeado)')
ax.set_ylabel('K verdadero')
ax.set_title('K   exactitud = %.3f' % metricas['K']['accuracy_rounded'])
fig.tight_layout()
guardar(fig, 'fig3_predicho_vs_verdadero')
plt.show()

In [ ]:
rng = np.random.default_rng(20261127)
B = 500
n = len(verdadero)
filas = []
for j, p in enumerate(PARAMS):
    v, q = verdadero[:, j].astype(float), predicho[:, j].astype(float)
    rs, rmses, accs = [], [], []
    for _ in range(B):
        k = rng.integers(0, n, n)
        rs.append(np.corrcoef(v[k], q[k])[0, 1])
        rmses.append(np.sqrt(np.mean((q[k] - v[k]) ** 2)))
        if p == 'K':
            accs.append(np.mean(np.clip(np.rint(q[k]), 2, 4) == v[k]))
    fila = {'parámetro': SIMBOLO[p],
            'r': round(metricas[p]['pearson_r'], 3),
            'r IC95': '[%.3f, %.3f]' % tuple(np.percentile(rs, [2.5, 97.5])),
            'RMSE': round(metricas[p]['rmse'], 4),
            'RMSE IC95': '[%.4f, %.4f]' % tuple(np.percentile(rmses, [2.5, 97.5])),
            'R2': round(metricas[p]['r2'], 3),
            'exactitud K': '',
            'exactitud IC95': ''}
    if p == 'K':
        fila['exactitud K'] = '%.3f' % metricas['K']['accuracy_rounded']
        fila['exactitud IC95'] = '[%.3f, %.3f]' % tuple(np.percentile(accs, [2.5, 97.5]))
    filas.append(fila)

tabla1 = pd.DataFrame(filas)
print('Agentes de prueba evaluados', metricas['alpha']['n'])
print(tabla1.to_string(index=False))
tabla1.to_csv(FIG / 'tabla1_metricas_prueba.csv', index=False)

## Figura 4. Importancia de cada métrica

Norma de los pesos de entrada de cada métrica en la primera capa, en la red de selección con la λ elegida. Una norma mayor indica que la red usa más esa métrica. Los colores indican el tamaño de conjunto (TC) al que pertenece. El asterisco marca las métricas que el group lasso eliminó con la λ más alta. Como la primera capa es compartida por las cinco salidas, la norma no dice para qué parámetro sirve cada métrica.

La Figura 4b muestra solo las 20 métricas más usadas, en un tamaño que cabe en una columna del reporte.

In [ ]:
elegida = [r for r in seleccion['resultados'] if r['lambda'] == seleccion['lambda_elegida']][0]
mas_fuerte = max(seleccion['resultados'], key=lambda r: r['lambda'])
normas = pd.Series(elegida['norma_por_caracteristica']).sort_values()
eliminadas = [k for k, v in mas_fuerte['norma_por_caracteristica'].items() if v == 0]

def grupo(nombre):
    for t in ('tc2', 'tc3', 'tc4'):
        if nombre.endswith('_' + t):
            return t
    return 'global'

COLOR_TC = {'tc2': 'C0', 'tc3': 'C1', 'tc4': 'C2', 'global': 'C7'}
LEYENDA_TC = [Patch(color=c, label='Global' if g == 'global' else 'TC ' + g[-1]) for g, c in COLOR_TC.items()]

def barras(ax, serie, tam):
    ax.barh(range(len(serie)), serie.values, color=[COLOR_TC[grupo(m)] for m in serie.index])
    ax.set_yticks(range(len(serie)))
    ax.set_yticklabels([m + ('  *' if m in eliminadas else '') for m in serie.index], fontsize=tam)
    ax.set_xlabel('Norma de los pesos de entrada (λ = %s)\n* eliminada con λ = %s'
                  % (seleccion['lambda_elegida'], mas_fuerte['lambda']))
    ax.legend(handles=LEYENDA_TC, frameon=False, loc='lower center', bbox_to_anchor=(0.5, 1.0), ncol=4)

fig, ax = plt.subplots(figsize=(6, 11))
barras(ax, normas, 7)
fig.tight_layout()
guardar(fig, 'fig4_importancia_metricas')
plt.show()

fig, ax = plt.subplots(figsize=(5, 5))
barras(ax, normas.tail(20), 8)
fig.tight_layout()
guardar(fig, 'fig4b_importancia_top20')
plt.show()

## Figura 5. Diferencia de parámetros contra conducta generada (parte B)

Es la figura central para el argumento de soluciones múltiples. Cada agente de prueba tiene parámetros verdaderos conocidos. Con los verdaderos y con los predichos por la red se simularon 1,000 secuencias en la secuencia de estímulos del agente, con los mismos números aleatorios, y se midió la coincidencia ensayo a ensayo con las decisiones del agente. La coincidencia con los parámetros verdaderos es el techo, el equivalente al acuerdo test-retest del agente consigo mismo.

En el eje x está cuánto se alejó la red de la verdad. En el eje y está la coincidencia de la red dividida entre la de la verdad (1 significa que generan conducta igual de parecida). Las líneas punteadas marcan 80, 90 y 95 % del techo. Muchos agentes con diferencia grande y valor cercano a 1 indican parámetros distintos que explican igual las mismas decisiones. La fila de abajo repite el análisis solo en los ensayos donde el agente se equivocó.

La tabla de abajo es descriptiva. Muestra el porcentaje de agentes con al menos 95 % del techo según qué tan lejos quedó la red.

In [ ]:
FILAS5 = [('coinc_rel', 'Coincidencia\nrelativa al techo'), ('coinc_err_rel', 'En errores\nrelativa al techo')]
fig, axs = plt.subplots(2, 5, figsize=(16, 6.4))
for f, (col, etiqueta) in enumerate(FILAS5):
    ylo, yhi = np.nanpercentile(sim[col], [0.5, 99.5])
    for j, p in enumerate(PARAMS):
        ax = axs[f, j]
        d = sim[['dif_' + p, col]].dropna()
        x = d['dif_' + p].abs()
        if p == 'K':
            pos = [k for k in (0, 1, 2) if (x == k).any()]
            ax.boxplot([d.loc[x == k, col] for k in pos], positions=pos, widths=0.5, showfliers=False)
            ax.set_xticks([0, 1, 2])
            ax.set_xticklabels(['0', '1', '2'])
            ax.set_xlim(-0.6, 2.6)
        else:
            ax.hexbin(x, d[col], gridsize=35, mincnt=1, bins='log', cmap='viridis', extent=(0, x.max(), ylo, yhi))
        for u in UMBRALES:
            ax.axhline(u, color='C3', lw=0.8, ls=':')
        ax.axhline(1.0, color='k', lw=0.8)
        ax.set_ylim(ylo, yhi)
        if f == 0:
            ax.set_title(SIMBOLO[p])
        if f == 1:
            ax.set_xlabel('|red − verdad| en ' + SIMBOLO[p])
        if j == 0:
            ax.set_ylabel(etiqueta)
fig.tight_layout()
guardar(fig, 'fig5_diferencia_vs_conducta')
plt.show()

BORDES = {'alpha': [0, 0.1, 0.2, 0.3, 1], 'phi': [0, 0.1, 0.2, 0.3, 1],
          'rho': [0, 0.1, 0.2, 0.3, 1], 'epsilon': [0, 0.05, 0.1, 0.15, 0.5]}
partes = []
for p in PARAMS:
    x = sim['dif_' + p].abs()
    g = x if p == 'K' else pd.cut(x, BORDES[p], include_lowest=True)
    t = sim.groupby(g, observed=False)['coinc_rel'].agg(n='size', pct_95=lambda s: 100 * (s >= 0.95).mean())
    t = t.reset_index().rename(columns={'dif_' + p: '|dif|'})
    t.insert(0, 'parámetro', SIMBOLO[p])
    t['|dif|'] = t['|dif|'].astype(str)
    partes.append(t)
tabla_b = pd.concat(partes, ignore_index=True)
print('Porcentaje de agentes con coincidencia >= 95 % del techo, según la distancia entre red y verdad')
print(tabla_b.round(1).to_string(index=False))

## Figura 5b. Versión compacta para el reporte

Resume la Figura 5 en una sola gráfica. En el eje x está la distancia entre la red y la verdad, expresada como porcentaje del rango de cada parámetro para poder comparar parámetros (α, ϕ y ρ van de 0.01 a 0.99, ε de 0.01 a 0.49 y K de 2 a 4). En el eje y está el porcentaje de agentes cuyos parámetros predichos alcanzan al menos 95 % del techo. Las barras son intervalos de confianza al 95 % de ese porcentaje. Una línea plana indica que alejarse de la verdad en ese parámetro no cambia la conducta (soluciones múltiples). Una línea que baja indica que sí la cambia (parámetro identificable). K solo aparece en los extremos porque su diferencia vale 0, 1 o 2.

In [ ]:
BINS_REL = [0, 0.1, 0.2, 0.3, 1.01]
ETQ = ['< 10 %', '10 a 20 %', '20 a 30 %', '> 30 %']
MARCA = {'alpha': 'o', 'phi': 's', 'rho': '^', 'epsilon': 'D', 'K': 'v'}
ok95 = sim['coinc_rel'] >= 0.95
filas = []
fig, ax = plt.subplots(figsize=(5.2, 3.4))
for k, p in enumerate(PARAMS):
    rel = sim['dif_' + p].abs() / RANGO[p]
    g = pd.cut(rel, BINS_REL, right=False, labels=False)
    pos, pct, ic = [], [], []
    for b in range(4):
        m = (g == b)
        nb = int(m.sum())
        if nb < 20:
            continue
        q = ok95[m].mean()
        pos.append(b + (k - 2) * 0.06)
        pct.append(100 * q)
        ic.append(100 * 1.96 * np.sqrt(q * (1 - q) / nb))
        filas.append({'parámetro': SIMBOLO[p], 'distancia': ETQ[b], 'n': nb, '% >= 95': round(100 * q, 1),
                      'IC95 ±': round(100 * 1.96 * np.sqrt(q * (1 - q) / nb), 1)})
    ax.errorbar(pos, pct, yerr=ic, marker=MARCA[p], ms=5, lw=1.2, capsize=2, label=SIMBOLO[p])
ax.set_xticks(range(4))
ax.set_xticklabels(ETQ)
ax.set_xlabel('Distancia entre red y verdad (% del rango del parámetro)')
ax.set_ylabel('% de agentes con ≥ 95 % del techo')
ax.legend(frameon=False, ncol=5, fontsize=8, loc='lower left')
fig.tight_layout()
guardar(fig, 'fig5b_compacta')
plt.show()
print(pd.DataFrame(filas).to_string(index=False))

## Figura 6 y Tabla 2. Distribución de la coincidencia relativa al techo

Histogramas de la coincidencia de la red dividida entre la del techo, en los 10,000 agentes simulados (techo = parámetros verdaderos) y en los 37 participantes (techo = MLE de la tesis). Las líneas punteadas marcan 80, 90 y 95 %. La Tabla 2 da la mediana, los cuartiles y el porcentaje que supera cada umbral. Las dos poblaciones no se comparan entre sí, porque tienen techos distintos y los participantes aprenden mucho mejor que el promedio de la rejilla.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 3.4))
for ax, (col, titulo) in zip(axs, [('coinc_rel', 'Coincidencia'), ('coinc_err_rel', 'Coincidencia en errores')]):
    s, r = sim[col].dropna(), real[col].dropna()
    lo = min(np.percentile(s, 0.5), r.min())
    hi = max(np.percentile(s, 99.5), r.max())
    bins = np.linspace(lo, hi, 50)
    ax.hist(s, bins=bins, density=True, alpha=0.5, color='C0', label='Simulados, techo = verdad (n=%d)' % len(s))
    ax.hist(r, bins=bins, density=True, alpha=0.5, color='C1', label='Reales, techo = MLE (n=%d)' % len(r))
    for u in UMBRALES:
        ax.axvline(u, color='C3', lw=0.8, ls=':')
    ax.axvline(1.0, color='k', lw=0.8)
    ax.set_xlabel(titulo + ' relativa al techo (red / techo)')
    ax.set_ylabel('Densidad')
axs[0].legend(frameon=False, fontsize=8)
fig.tight_layout()
guardar(fig, 'fig6_distribucion_relativa')
plt.show()

filas = []
for nombre, df in [('Simulados', sim), ('Reales', real)]:
    for col in ['coinc_rel', 'coinc_err_rel']:
        x = df[col].dropna()
        fila = {'grupo': nombre, 'medida': col, 'n': len(x), 'mediana': x.median(),
                'Q1': x.quantile(0.25), 'Q3': x.quantile(0.75)}
        for u in UMBRALES:
            fila['%% >= %d' % round(100 * u)] = 100 * (x >= u).mean()
        filas.append(fila)
tabla2 = pd.DataFrame(filas)
print(tabla2.round(3).to_string(index=False))
tabla2.to_csv(FIG / 'tabla2_umbrales.csv', index=False)

## Figura 7. Participantes reales, parámetros MLE contra parámetros de la red

Cada punto es un participante (P01 a P37). La línea punteada es la identidad. Aquí no hay verdad, así que la figura solo muestra cuánto difieren los dos métodos. Los parámetros de la red están recortados al rango de entrenamiento y K está redondeado. En K se agrega un pequeño desplazamiento al azar para que no se encimen los puntos.

In [ ]:
rng7 = np.random.default_rng(0)
fig, axs = plt.subplots(1, 5, figsize=(16, 3.3))
for j, p in enumerate(PARAMS):
    ax = axs[j]
    xm, yr = real[p + '_mle'].astype(float), real[p + '_red'].astype(float)
    xs, ys = xm.copy(), yr.copy()
    if p == 'K':
        xs = xs + rng7.uniform(-0.12, 0.12, len(xs))
        ys = ys + rng7.uniform(-0.12, 0.12, len(ys))
        lo, hi = 1.6, 4.4
    else:
        lo, hi = 0, (0.5 if p == 'epsilon' else 1)
    ax.scatter(xs, ys, s=20, color='C1', edgecolor='k', lw=0.3, alpha=0.85)
    ax.plot([lo, hi], [lo, hi], color='0.5', lw=1, ls='--')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    rr = np.corrcoef(xm, yr)[0, 1] if (xm.std() > 0 and yr.std() > 0) else np.nan
    ax.set_title('%s   r = %.2f' % (SIMBOLO[p], rr))
    ax.set_xlabel('MLE (tesis)')
    if j == 0:
        ax.set_ylabel('Red')
fig.tight_layout()
guardar(fig, 'fig7_reales_mle_vs_red')
plt.show()

## Figura 8. Participantes reales, coincidencia con el techo contra coincidencia con la red

Cada punto es un participante. En el eje x está la coincidencia de las decisiones simuladas con sus parámetros MLE y en el eje y con los de la red. Las barras son la desviación estándar entre las 1,000 simulaciones (la variación de una simulación individual, no un intervalo de confianza). La línea negra es la igualdad y las punteadas marcan 80, 90 y 95 % del techo. Puntos cerca de la línea negra indican que los dos conjuntos de parámetros generan conducta igual de parecida a la del participante.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4.2))
for ax, (base, titulo) in zip(axs, [('coinc', 'Coincidencia'), ('coinc_err', 'Coincidencia en errores')]):
    x, y = real[base + '_mean_mle'], real[base + '_mean_red']
    ax.errorbar(x, y, xerr=real[base + '_sd_mle'], yerr=real[base + '_sd_red'], fmt='o', ms=4,
                color='C1', ecolor='0.75', elinewidth=0.8, capsize=0)
    lo = max(0, min(x.min(), y.min()) - 0.1)
    hi = min(1, max(x.max(), y.max()) + 0.1)
    t = np.linspace(lo, hi, 2)
    ax.plot(t, t, color='k', lw=1, label='Igual al techo')
    for u in UMBRALES:
        ax.plot(t, u * t, color='C3', lw=0.8, ls=':')
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_xlabel(titulo + ' con parámetros MLE (techo)')
    ax.set_ylabel(titulo + ' con parámetros de la red')
axs[0].legend(frameon=False)
fig.tight_layout()
guardar(fig, 'fig8_reales_coincidencia')
plt.show()

## Figura 9. Compensaciones entre parámetros en los errores de la red (agentes simulados)

Pair plot del error de la red (red − verdad) en cada parámetro, en la muestra de 200,000 agentes de prueba. Las predicciones se recortaron al rango de entrenamiento y K se redondeó, igual que en la parte B. Debajo de la diagonal, cada panel cruza el error en un parámetro (eje x) con el error en otro (eje y). En la diagonal está el histograma de cada error. Arriba de la diagonal está la r de cada par (en rojo si |r| ≥ 0.3).

Una nube inclinada hacia abajo (r negativa) indica compensación. Cuando la red sobrestima un parámetro tiende a subestimar el otro, lo que sugiere que subir uno y bajar el otro produce conducta parecida. Una nube redonda (r cercana a 0) indica que los errores de ese par son independientes.

In [ ]:
def recortar(P):
    P = np.array(P, dtype=float, copy=True)
    for j, p in enumerate(PARAMS):
        if p == 'K':
            P[:, j] = np.rint(P[:, j])
        P[:, j] = np.clip(P[:, j], CLIP[p][0], CLIP[p][1])
    return P

def pair_plot(D, titulo_eje, nombre, puntos=False):
    R = np.corrcoef(D, rowvar=False)
    Dj = D.copy()
    if puntos:
        Dj[:, 4] = Dj[:, 4] + np.random.default_rng(1).uniform(-0.1, 0.1, len(Dj))
    fig, axs = plt.subplots(5, 5, figsize=(11, 11))
    for i in range(5):
        for j in range(5):
            ax = axs[i, j]
            if i == j:
                ax.hist(D[:, i], bins=15 if puntos else 60, color='C0')
                ax.set_yticks([])
            elif i > j:
                if puntos:
                    ax.scatter(Dj[:, j], Dj[:, i], s=14, color='C1', edgecolor='k', lw=0.3)
                else:
                    ax.hexbin(D[:, j], D[:, i], gridsize=35, mincnt=1, bins='log', cmap='viridis')
                ax.axhline(0, color='0.5', lw=0.5)
                ax.axvline(0, color='0.5', lw=0.5)
            else:
                ax.axis('off')
                ax.text(0.5, 0.5, 'r = %.2f' % R[i, j], ha='center', va='center', fontsize=12,
                        color='C3' if abs(R[i, j]) >= 0.3 else '0.3')
            if i == 4:
                ax.set_xlabel(titulo_eje + ' ' + SIMBOLO[PARAMS[j]])
            if j == 0 and i > 0:
                ax.set_ylabel(titulo_eje + ' ' + SIMBOLO[PARAMS[i]])
    fig.tight_layout()
    guardar(fig, nombre)
    plt.show()
    tabla = pd.DataFrame(R, index=[SIMBOLO[p] for p in PARAMS], columns=[SIMBOLO[p] for p in PARAMS])
    print('Correlación entre errores')
    print(tabla.round(3).to_string())
    return tabla

err_sim = recortar(predicho) - verdadero
r_err_sim = pair_plot(err_sim, 'error en', 'fig9_pairplot_errores_simulados')

## Figura 10. Diferencias entre red y MLE en los participantes reales

El mismo pair plot con los 37 participantes. Aquí no hay verdad, así que se grafica la diferencia red − MLE en cada parámetro. Cada punto es un participante. En K se agrega un pequeño desplazamiento al azar para que no se encimen los puntos (la r se calcula sin ese desplazamiento). Con 37 personas, una r necesita pasar de aproximadamente 0.32 en valor absoluto para ser distinta de cero con 95 % de confianza.

In [ ]:
dif_real = real[['dif_' + p for p in PARAMS]].to_numpy(dtype=float)
r_dif_real = pair_plot(dif_real, 'red − MLE en', 'fig10_pairplot_reales', puntos=True)

## Figura 11 y Tabla 3. Redundancia entre las 59 métricas

Correlación de Pearson entre cada par de métricas, en 200,000 agentes al azar. Antes se rellenaron los datos faltantes con la mediana, igual que hizo la red. Las métricas están ordenadas con un agrupamiento jerárquico para que las que se parecen queden juntas, así que los bloques de color intenso indican grupos de métricas redundantes. Rojo es correlación positiva y azul negativa.

Debajo se cuentan los pares con correlación alta, se listan los 20 pares más correlacionados (Tabla 3) y se agrupan las métricas cuya correlación promedio, en valor absoluto, pasa de 0.9 y de 0.8. El número de grupos indica con cuántas métricas distintas se quedaría uno si se dejara una sola por grupo. El agrupamiento usa el valor absoluto, así que dos métricas con correlación muy negativa también cuentan como redundantes.

In [ ]:
from scipy.cluster.hierarchy import linkage, leaves_list, fcluster
from scipy.spatial.distance import squareform

if Xm is None:
    print('Falta muestra_metricas.npz en la carpeta. Súbelo y vuelve a correr desde la celda de carga.')
else:
    med_m = np.nanmedian(Xm, axis=0)
    Xi = np.where(np.isnan(Xm), med_m, Xm)
    usar = Xi.std(axis=0) > 0
    if not usar.all():
        print('Métricas sin variación (se excluyen):', [m for m, u in zip(nombres_m, usar) if not u])
    Xi = Xi[:, usar]
    nom = [m for m, u in zip(nombres_m, usar) if u]
    C = np.corrcoef(Xi, rowvar=False)

    dist = 1 - np.abs(C)
    np.fill_diagonal(dist, 0)
    Z = linkage(squareform(dist, checks=False), method='average')
    orden = leaves_list(Z)

    fig, ax = plt.subplots(figsize=(10, 9))
    im = ax.imshow(C[np.ix_(orden, orden)], cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(nom)))
    ax.set_xticklabels([nom[k] for k in orden], rotation=90, fontsize=5)
    ax.set_yticks(range(len(nom)))
    ax.set_yticklabels([nom[k] for k in orden], fontsize=5)
    fig.colorbar(im, ax=ax, fraction=0.04, label='Correlación de Pearson')
    fig.tight_layout()
    guardar(fig, 'fig11_correlacion_metricas')
    plt.show()

    iu = np.triu_indices(len(nom), 1)
    a = np.abs(C[iu])
    for u in (0.9, 0.8, 0.7):
        print('Pares con |r| > %.1f: %d de %d' % (u, int((a > u).sum()), len(a)))

    top = np.argsort(-a)[:20]
    tabla3 = pd.DataFrame([{'métrica 1': nom[iu[0][k]], 'métrica 2': nom[iu[1][k]], 'r': C[iu[0][k], iu[1][k]]}
                           for k in top]).round(3)
    print()
    print('Tabla 3. Los 20 pares de métricas más correlacionados')
    print(tabla3.to_string(index=False))
    tabla3.to_csv(FIG / 'tabla3_pares_correlacionados.csv', index=False)

    for u in (0.9, 0.8):
        grupos = fcluster(Z, t=1 - u, criterion='distance')
        print()
        print('Con correlación promedio |r| > %.1f quedan %d grupos de %d métricas. Grupos con más de una:'
              % (u, grupos.max(), len(nom)))
        for g in np.unique(grupos):
            miembros = [nom[k] for k in np.where(grupos == g)[0]]
            if len(miembros) > 1:
                print('   ', ', '.join(miembros))

## Figura 12. Cuántas dimensiones independientes hay en las métricas

Análisis de componentes principales sobre la misma matriz de correlación. En el eje x está el número de componentes y en el eje y la varianza acumulada que explican. Si pocas componentes explican casi toda la varianza, las 59 métricas contienen en realidad pocas dimensiones de información distintas. La celda imprime cuántas componentes hacen falta para 80, 90, 95 y 99 %, y el número efectivo de dimensiones (participation ratio), que resume en un solo número cuántas dimensiones pesan de verdad.

In [ ]:
if Xm is not None:
    ev = np.clip(np.sort(np.linalg.eigvalsh(C))[::-1], 0, None)
    acum = 100 * np.cumsum(ev) / ev.sum()
    fig, ax = plt.subplots(figsize=(5, 3.2))
    ax.plot(np.arange(1, len(ev) + 1), acum, marker='o', ms=3, color='C0')
    for u in (80, 90, 95, 99):
        k = int(np.searchsorted(acum, u) + 1)
        ax.axhline(u, color='0.7', lw=0.6, ls=':')
        print('Componentes para explicar %d %% de la varianza: %d de %d' % (u, k, len(ev)))
    ax.set_xlabel('Número de componentes principales')
    ax.set_ylabel('Varianza explicada acumulada (%)')
    fig.tight_layout()
    guardar(fig, 'fig12_componentes_principales')
    plt.show()
    print('Número efectivo de dimensiones (participation ratio): %.1f' % (ev.sum() ** 2 / (ev ** 2).sum()))